In [ ]:
# ==============================================================================
# SECTION 1: WHY NUMPY EXISTS & THE SPEED GAP
# ==============================================================================
# In standard Python:
#   A list is an array of POINTERS to boxed objects scattered across heap memory.
# In NumPy:
#   An ndarray is a CONTIGUOUS block of raw, uniform primitive C-types in memory.
#   This enables CPU cache locality, SIMD vectorisation, and parallel processing.

import numpy as np

print(f"NumPy Version: {np.__version__}")

# Benchmark: Python loop vs. Vectorised NumPy operation (1,000,000 elements)
size = 1_000_000
py_list = list(range(size))
np_arr = np.arange(size)
print(np_arr)

NumPy Version: 2.1.3
[     0      1      2 ... 999997 999998 999999]


In [ ]:
# ==============================================================================
# SECTION 2: ARRAY CREATION & ATTRIBUTES
# ==============================================================================
# Seed for reproducibility
np.random.seed(42)

# 1. Creation from lists
a1 = np.array([1, 2, 3, 4], dtype=np.float32)

# 2. Structural initialisations
zeros = np.zeros((2, 3))               # 2x3 matrix of 0.0
ones = np.ones((2, 2), dtype=int)      # 2x2 matrix of 1s
full = np.full((2, 3), fill_value=7)   # 2x3 matrix filled with 7
identity = np.eye(3)                   # 3x3 Identity matrix

# 3. Range-based creation
seq_step = np.arange(0, 10, 2)         # start, stop (exclusive), step -> [0, 2, 4, 6, 8]
seq_lin = np.linspace(0, 1, 5)         # start, stop (inclusive), num points -> [0., 0.25, 0.5, 0.75, 1.]

# 4. Random arrays
rand_uniform = np.random.rand(2, 3)    # Uniform [0, 1)
rand_normal = np.random.randn(2, 3)    # Standard normal (mean 0, std 1)
rand_int = np.random.randint(1, 10, size=(2, 3)) # Discrete integers [low, high)

print("Array:\n", rand_int)
print("Attributes:")
print(f" - shape (dimensions):        {rand_int.shape}")
print(f" - ndim  (num of axes):       {rand_int.ndim}")
print(f" - size  (total elements):    {rand_int.size}")
print(f" - dtype (element data type): {rand_int.dtype}")

Array:
 [[6 9 1]
 [3 7 4]]
Attributes:
 - shape (dimensions):        (2, 3)
 - ndim  (num of axes):       2
 - size  (total elements):    6
 - dtype (element data type): int64


In [ ]:
# ==============================================================================
# SECTION 3: INDEXING, SLICING, & THE "VIEW VS COPY" GOTCHA
# ==============================================================================
mat = np.arange(1, 13).reshape(3, 4)
print("Original Matrix:\n", mat)

# Basic 2D Slicing: [rows, cols]
sub_mat = mat[0:2, 1:3]
print("\nSlice [0:2, 1:3]:\n", sub_mat)

# --- CRITICAL GOTCHA: Slices create VIEWS (shared memory), NOT deep copies! ---
sub_mat[0, 0] = 999
print("\nAfter modifying sub_mat[0, 0] to 999:")
print("Original Matrix is mutated!:\n", mat)

# To prevent mutating the original, explicitly use .copy()
mat_copy = mat[0:2, 1:3].copy()
mat_copy[0, 0] = -1
print("\nModifying an explicit copy does NOT mutate original:")
print("Original mat[0, 1] remains:", mat[0, 1])

Original Matrix:
 [[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]

Slice [0:2, 1:3]:
 [[2 3]
 [6 7]]

After modifying sub_mat[0, 0] to 999:
Original Matrix is mutated!:
 [[  1 999   3   4]
 [  5   6   7   8]
 [  9  10  11  12]]

Modifying an explicit copy does NOT mutate original:
Original mat[0, 1] remains: 999


In [ ]:
# ==============================================================================
# SECTION 4: BOOLEAN MASKING & FANCY INDEXING
# ==============================================================================
arr = np.array([10, 25, 30, 45, 50, 65])

# 1. Boolean Masking (filtering)
mask = arr > 30
print("Boolean Mask (arr > 30):", mask)
print("Filtered Elements:       ", arr[mask])

# In-place condition replacement
arr[arr > 40] = 0
print("After replacing values > 40 with 0:", arr)

# 2. Fancy Indexing (integer array indexing - always returns a COPY)
data = np.arange(10, 100, 10).reshape(3, 3)
print("\nData:\n", data)
rows = np.array([0, 2])
cols = np.array([1, 2])
print("Selected elements (0,1) and (2,2):", data[rows, cols])

Boolean Mask (arr > 30): [False False False  True  True  True]
Filtered Elements:        [45 50 65]
After replacing values > 40 with 0: [10 25 30  0  0  0]

Data:
 [[10 20 30]
 [40 50 60]
 [70 80 90]]
Selected elements (0,1) and (2,2): [20 90]


In [ ]:
# ==============================================================================
# SECTION 5: SHAPE MANIPULATION & EXPANSIONS
# ==============================================================================
arr = np.arange(6) # [0, 1, 2, 3, 4, 5]

# 1. Reshape
r_arr = arr.reshape(2, 3)
print("Reshaped (2, 3):\n", r_arr)

# 2. Ravel (View where possible) vs Flatten (Always creates a Copy)
rav = r_arr.ravel()
flat = r_arr.flatten()

# 3. Transpose
print("Transposed (3, 2):\n", r_arr.T)

# 4. Dimension Expansion (used frequently in deep learning and broadcasting)
v = np.array([1, 2, 3]) # shape: (3,)
v_col1 = np.expand_dims(v, axis=1) # shape: (3, 1)
v_col2 = v[:, np.newaxis]          # shape: (3, 1)
print("Vector expanded to column shape:", v_col1.shape)

Reshaped (2, 3):
 [[0 1 2]
 [3 4 5]]
Transposed (3, 2):
 [[0 3]
 [1 4]
 [2 5]]
Vector expanded to column shape: (3, 1)


In [ ]:
# ==============================================================================
# SECTION 7: AGGREGATIONS & THE AXIS PARAMETER
# ==============================================================================
# Visual rule for 2D:
# axis=0: Collapse rows (move down vertically -> compute for each column)
# axis=1: Collapse columns (move across horizontally -> compute for each row)

X = np.array([
    [10, 20, 30],
    [40, 50, 60]
])

print("Data:\n", X)
print("Sum Total:           ", np.sum(X))
print("Sum axis=0 (per col):", np.sum(X, axis=0))
print("Sum axis=1 (per row):", np.sum(X, axis=1))
print("Mean per column:     ", np.mean(X, axis=0))
print("Std per row:         ", np.std(X, axis=1))
print("Index of max overall:", np.argmax(X))
print("Index of max per col:", np.argmax(X, axis=0))

Data:
 [[10 20 30]
 [40 50 60]]
Sum Total:            210
Sum axis=0 (per col): [50 70 90]
Sum axis=1 (per row): [ 60 150]
Mean per column:      [25. 35. 45.]
Std per row:          [8.16496581 8.16496581]
Index of max overall: 5
Index of max per col: [1 1 1]


In [ ]:
# ==============================================================================
# LIVE CODE BENCHMARK: COLUMN-WISE NORMALISATION (Z-Score Standardisation)
# Formula: Z = (X - mean) / std
# ==============================================================================
# Matrix: 10,000 rows (samples), 100 columns (features)
N_ROWS, N_COLS = 10_000, 100
dataset = np.random.randn(N_ROWS, N_COLS) * 5 + 20

def python_loop_normalise(mat):
    """Normalises each column using standard nested Python loops."""
    rows, cols = mat.shape
    out = np.empty((rows, cols))
    for j in range(cols):
        # Calculate mean & std for column j
        col_sum = 0.0
        for i in range(rows):
            col_sum += mat[i, j]
        col_mean = col_sum / rows

        variance_sum = 0.0
        for i in range(rows):
            variance_sum += (mat[i, j] - col_mean) ** 2
        col_std = (variance_sum / rows) ** 0.5

        # Normalise column
        for i in range(rows):
            out[i, j] = (mat[i, j] - col_mean) / col_std
    return out

def vectorised_normalise(mat):
    """Normalises all columns simultaneously via axis aggregation & broadcasting."""
    mean = np.mean(mat, axis=0) # shape: (100,)
    std = np.std(mat, axis=0)   # shape: (100,)
    return (mat - mean) / std   # (10000, 100) - (100,) -> broadcasted!

# Validate equivalence
norm_loop = python_loop_normalise(dataset[:100, :10])
norm_vec = vectorised_normalise(dataset[:100, :10])
assert np.allclose(norm_loop, norm_vec)
print("Outputs match perfectly!\n")

print("--- Benchmark: Normalising (10,000 x 100) Matrix ---")
print("1. Nested Python Loops:")
%timeit python_loop_normalise(dataset)

print("2. Vectorised Broadcasting:")
%timeit vectorised_normalise(dataset)

Outputs match perfectly!

--- Benchmark: Normalising (10,000 x 100) Matrix ---
1. Nested Python Loops:
1.57 s ± 832 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
2. Vectorised Broadcasting:
10.8 ms ± 1.45 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [ ]:
# ==============================================================================
# SECTION 10: 12 STUDENT EXERCISES
# ==============================================================================

# --- Exercise 1 ---
# Create a 1D array of 20 evenly spaced values between 5 and 50 (inclusive).
# Hint: np.linspace
# YOUR CODE HERE:
ex1 = np.linspace(5, 50, 20)

# --- Exercise 2 ---
# Create a 4x4 matrix where border elements are 1 and inside elements are 0.
# YOUR CODE HERE:
ex2 = np.ones((4, 4))
ex2[1:-1, 1:-1] = 0

# --- Exercise 3 ---
# Create an array of random integers between 10 and 100 of shape (5, 5).
# Replace all even values with -1 without modifying the original array (return a new array).
# Hint: np.where
# YOUR CODE HERE:
orig = np.random.randint(10, 100, size=(5, 5))
ex3 = np.where(orig % 2 == 0, -1, orig)

# --- Exercise 4 ---
# Given a 1D array, extract elements that are greater than 20 AND divisible by 3.
# Hint: bitwise operator &
ex4_arr = np.array([12, 21, 24, 30, 5, 18, 45, 9, 33])
# YOUR CODE HERE:
ex4 = ex4_arr[(ex4_arr > 20) & (ex4_arr % 3 == 0)]

# --- Exercise 5 ---
# Given a 2D array, reverse its rows (top row becomes bottom row) using slicing.
ex5_mat = np.arange(16).reshape(4, 4)
# YOUR CODE HERE:
ex5 = ex5_mat[::-1, :]

# --- Exercise 6 ---
# Given an array of shape (6, 1) and an array of shape (1, 4), perform an addition
# that outputs an array of shape (6, 4).
# YOUR CODE HERE:
u = np.arange(6).reshape(6, 1)
v = np.arange(4).reshape(1, 4)
ex6 = u + v

# --- Exercise 7 ---
# Given a 2D array (5x3), subtract the mean of each column from the respective column.
# YOUR CODE HERE:
mat_ex7 = np.random.rand(5, 3)
ex7 = mat_ex7 - np.mean(mat_ex7, axis=0)

# --- Exercise 8 ---
# Given a 2D array (4x5), subtract the mean of each row from the respective row.
# Hint: keepdims=True or expand_dims
# YOUR CODE HERE:
mat_ex8 = np.random.rand(4, 5)
ex8 = mat_ex8 - np.mean(mat_ex8, axis=1, keepdims=True)

# --- Exercise 9 ---
# Find the row index of the maximum value in each column of a 2D array.
# YOUR CODE HERE:
mat_ex9 = np.array([[10, 50, 30], [40, 20, 90], [15, 80, 25]])
ex9 = np.argmax(mat_ex9, axis=0)

# --- Exercise 10 ---
# Stack two 1D arrays of length 4 as columns into a 2D array of shape (4, 2).
# Hint: np.column_stack or np.vstack with transpose
# YOUR CODE HERE:
c1 = np.array([1, 2, 3, 4])
c2 = np.array([5, 6, 7, 8])
ex10 = np.column_stack((c1, c2))

# --- Exercise 11 ---
# Replace all NaN values in a given 1D float array with the mean of non-NaN values.
# Hint: np.isnan(), np.nanmean()
ex11_arr = np.array([1.0, 2.0, np.nan, 4.0, np.nan, 6.0])
# YOUR CODE HERE:
mean_val = np.nanmean(ex11_arr)
ex11 = ex11_arr.copy()
ex11[np.isnan(ex11)] = mean_val

# --- Exercise 12 ---
# Min-Max Feature Scaling: Normalise a 2D matrix column-wise so all values lie in [0, 1].
# Formula: (X - min) / (max - min)
# YOUR CODE HERE:
X_raw = np.random.randint(1, 100, size=(6, 3)).astype(float)
col_min = np.min(X_raw, axis=0)
col_max = np.max(X_raw, axis=0)
ex12 = (X_raw - col_min) / (col_max - col_min)

print("All 12 exercises executed successfully!")

All 12 exercises executed successfully!
